# 02 — Preprocessing

Transforms raw data from `01_data_collection` into training-ready formats:
1. **Semantic corpus** — anime text documents for MLM pre-training
2. **Triplets** — (anchor, positive, negative) review pairs for contrastive learning
3. **CF matrix** — sparse user×anime rating matrix for the autoencoder

**Requires**: Run `01_data_collection.ipynb` first.

In [1]:
import json
import random
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path
from collections import defaultdict
from scipy import sparse

DATA_DIR = Path("data")
random.seed(42)
np.random.seed(42)

print("Loading collected data...")

Loading collected data...


In [2]:
# Load all data files from notebook 01
ratings_df = pd.read_csv(DATA_DIR / "ratings_filtered.csv")

with open(DATA_DIR / "id_map.json", "r", encoding="utf-8") as f:
    id_map = json.load(f)
mal_to_anilist = {int(k): v for k, v in id_map["mal_to_anilist"].items()}
anilist_to_mal = {int(k): v for k, v in id_map["anilist_to_mal"].items()}

metadata_path = DATA_DIR / "anilist_anime.jsonl"
catalog_fingerprint_path = DATA_DIR / "catalog_snapshot_fingerprint.json"

def _norm_text(v):
    return v.strip() if isinstance(v, str) and v.strip() else None

def _norm_str_list(v):
    if not isinstance(v, list):
        return []
    out = []
    for item in v:
        if isinstance(item, str) and item.strip():
            out.append(item.strip())
    return out

if not metadata_path.exists():
    snapshots_root = DATA_DIR / "catalog_snapshots"
    candidates = [p for p in snapshots_root.glob("catalog_snapshot_*") if (p / "anime_catalog.jsonl").exists()]
    if not candidates:
        raise FileNotFoundError(
            f"Missing {metadata_path} and no catalog snapshots found under {snapshots_root}."
        )
    latest_snapshot = sorted(candidates, key=lambda p: p.name)[-1]
    catalog_path = latest_snapshot / "anime_catalog.jsonl"
    generated = 0
    with open(catalog_path, "r", encoding="utf-8") as src, open(metadata_path, "w", encoding="utf-8") as dst:
        for line in src:
            if not line.strip():
                continue
            c = json.loads(line)
            aid = c.get("anilist_id")
            if not isinstance(aid, int) or aid <= 0:
                continue
            meta = c.get("metadata_json") if isinstance(c.get("metadata_json"), dict) else {}
            title_romaji = _norm_text(c.get("title_romaji")) or _norm_text(meta.get("title_romaji"))
            title_english = _norm_text(c.get("title_english")) or _norm_text(meta.get("title_english"))
            title_native = _norm_text(c.get("title_native")) or _norm_text(meta.get("title_native"))
            title = title_english or title_romaji or title_native or ""
            synonyms = _norm_str_list(meta.get("synonyms") or meta.get("aliases"))
            genres = _norm_str_list(c.get("genres") if isinstance(c.get("genres"), list) else meta.get("genres"))
            tags = meta.get("tags") if isinstance(meta.get("tags"), list) else []
            studios = meta.get("studios") if isinstance(meta.get("studios"), list) else []
            relations = meta.get("relations") if isinstance(meta.get("relations"), list) else []
            popularity = c.get("anilist_popularity")
            if not isinstance(popularity, int):
                pop2 = meta.get("popularity")
                popularity = pop2 if isinstance(pop2, int) else None
            row = {
                "anilist_id": aid,
                "mal_id": c.get("mal_id"),
                "title": title,
                "title_romaji": title_romaji,
                "title_english": title_english,
                "title_native": title_native,
                "synonyms": synonyms,
                "aliases": synonyms,
                "genres": genres,
                "tags": tags,
                "studios": studios,
                "relations": relations,
                "description": _norm_text(c.get("description")) or _norm_text(meta.get("description")),
                "average_score": c.get("average_score"),
                "popularity": popularity,
                "anilist_popularity": popularity,
                "episodes": c.get("episodes"),
                "format": _norm_text(c.get("format")),
                "status": _norm_text(c.get("status")),
                "season": _norm_text(c.get("season")),
                "season_year": c.get("season_year"),
                "is_adult": c.get("is_adult"),
                "cover_image": c.get("cover_image") or meta.get("cover_image"),
                "metadata_fingerprint": c.get("metadata_fingerprint"),
                "updated_at": c.get("updated_at"),
            }
            dst.write(json.dumps(row, ensure_ascii=False) + "\n")
            generated += 1
    print(f"Generated {generated:,} metadata rows from snapshot: {latest_snapshot.name}")

anime_metadata = []
with open(metadata_path, "r", encoding="utf-8") as f:
    for line in f:
        anime_metadata.append(json.loads(line))
anime_by_id = {a["anilist_id"]: a for a in anime_metadata}

reviews = []
with open(DATA_DIR / "anilist_reviews.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        reviews.append(json.loads(line))

print(f"Ratings: {len(ratings_df):,}")
print(f"Anime metadata: {len(anime_metadata):,} (source: {metadata_path.name})")
print(f"Reviews: {len(reviews):,}")
print(f"ID mappings: {len(mal_to_anilist):,}")
if catalog_fingerprint_path.exists():
    with open(catalog_fingerprint_path, "r", encoding="utf-8") as f:
        catalog_fp = json.load(f)
    src = catalog_fp.get("source", {})
    out = catalog_fp.get("output", {})
    print(
        "Catalog fingerprint loaded:",
        f"rows={out.get('rows', 'n/a')}",
        f"manifest={src.get('snapshot_manifest', 'n/a')}"
    )

Ratings: 57,178,041
Anime metadata: 5,000
Reviews: 12,481
ID mappings: 18,622


## 1. Semantic Corpus

For each anime, build a rich text document combining:
- Title + genres + tags
- Synopsis (from Kaggle)
- Reviews (from AniList)

Truncated to ~2048 tokens per anime.

In [3]:
# Load Kaggle synopses and index by MAL ID
KAGGLE_INPUT = Path("/kaggle/input/anime-recommendation-database-2020")
if not KAGGLE_INPUT.exists():
    KAGGLE_INPUT = DATA_DIR / "kaggle"

synopsis_df = pd.read_csv(KAGGLE_INPUT / "anime_with_synopsis.csv")
synopsis_by_mal = dict(zip(synopsis_df["MAL_ID"], synopsis_df["sypnopsis"]))
print(f"Synopses loaded: {len(synopsis_by_mal):,}")

Synopses loaded: 16,214


In [4]:
# Import from a real module so preprocessing logic is reusable and testable.
import sys
from pathlib import Path
if not Path("semantic_preprocessing.py").exists() and Path("notebooks/semantic_preprocessing.py").exists():
    sys.path.append(str(Path("notebooks").resolve()))
from semantic_preprocessing import preprocess_review_text

# Group reviews by AniList ID using high-signal sentence extraction.
MASK_TITLE_PROBABILITY = 0.45
REVIEW_MAX_CHARS = 1200
REVIEW_MIN_OUTPUT_CHARS = 140

reviews_by_anime = defaultdict(list)
review_meta_by_anime = defaultdict(list)
raw_lengths = []
processed_lengths = []
review_diag_totals = {
    "total_reviews": 0,
    "kept_reviews": 0,
    "dropped_short": 0,
    "dropped_noise": 0,
    "masked_reviews": 0,
    "total_sentences": 0,
    "noise_filtered": 0,
    "low_score_filtered": 0,
    "kept_sentences": 0,
}


def _anime_title(aid):
    anime = anime_by_id.get(aid)
    if not anime:
        return ""
    return str(anime.get("title") or "").strip()


for r in reviews:
    aid = r.get("anilist_id")
    if aid is None:
        continue

    body = r.get("body", "")
    if isinstance(body, str):
        raw_lengths.append(len(body))

    review_diag_totals["total_reviews"] += 1
    cleaned, diag = preprocess_review_text(
        body,
        anime_title=_anime_title(aid),
        anime_id=aid,
        max_chars=REVIEW_MAX_CHARS,
        min_output_chars=REVIEW_MIN_OUTPUT_CHARS,
        mask_title_probability=MASK_TITLE_PROBABILITY,
        max_sentences=10,
        min_score=0.9,
        min_sentence_chars=35,
        return_diagnostics=True,
    )

    review_diag_totals["kept_reviews"] += int(diag.get("kept_reviews", 0))
    review_diag_totals["dropped_short"] += int(diag.get("dropped_short", 0))
    review_diag_totals["dropped_noise"] += int(diag.get("dropped_noise", 0))
    review_diag_totals["masked_reviews"] += int(bool(diag.get("mask_applied")))
    review_diag_totals["total_sentences"] += int(diag.get("total_sentences", 0))
    review_diag_totals["noise_filtered"] += int(diag.get("noise_filtered", 0))
    review_diag_totals["low_score_filtered"] += int(diag.get("low_score_filtered", 0))
    review_diag_totals["kept_sentences"] += int(diag.get("kept_sentences", 0))

    if cleaned:
        reviews_by_anime[aid].append(cleaned)
        review_meta_by_anime[aid].append({
            "text": cleaned,
            "mask_applied": bool(diag.get("mask_applied")),
        })
        processed_lengths.append(len(cleaned))

print(f"Anime with reviews: {len(reviews_by_anime):,}")
review_counts = [len(v) for v in reviews_by_anime.values()]
print(f"Reviews per anime: min={min(review_counts)}, median={np.median(review_counts):.0f}, max={max(review_counts)}")

if raw_lengths and processed_lengths:
    print(
        f"Review char lengths: raw_median={np.median(raw_lengths):.0f}, "
        f"processed_median={np.median(processed_lengths):.0f}"
    )

masked_ratio = (
    review_diag_totals["masked_reviews"] / review_diag_totals["total_reviews"]
    if review_diag_totals["total_reviews"] else 0.0
)
print(
    "Review preprocessing diagnostics:",
    {
        "kept_reviews": review_diag_totals["kept_reviews"],
        "dropped_short": review_diag_totals["dropped_short"],
        "dropped_noise": review_diag_totals["dropped_noise"],
        "masked_ratio": round(masked_ratio, 4),
    },
)



Anime with reviews: 2,796
Reviews per anime: min=1, median=2, max=25
Review char lengths: raw_median=4779, processed_median=1200
Review preprocessing diagnostics: {'kept_reviews': 12427, 'dropped_short': 5, 'dropped_noise': 49, 'masked_ratio': 0.4495}


In [5]:
import difflib

MAX_CHARS = 9000
SYNOPSIS_CHAR_CAP = 1800
MAX_REVIEWS_PER_ANIME = 6


def _as_nonempty_str(value):
    if value is None:
        return None
    s = str(value).strip()
    if not s or s.lower() == "nan":
        return None
    return s


def _normalize_str_list(value):
    out = []
    if isinstance(value, list):
        for item in value:
            if isinstance(item, str):
                text = item.strip()
                if text:
                    out.append(text)
    elif isinstance(value, str):
        text = value.strip()
        if text:
            out.append(text)
    return out


def _extract_aliases(anime):
    aliases = []
    aliases.extend(_normalize_str_list(anime.get("synonyms")))
    aliases.extend(_normalize_str_list(anime.get("aliases")))
    dedup = []
    seen = set()
    for alias in aliases:
        key = alias.lower()
        if key in seen:
            continue
        seen.add(key)
        dedup.append(alias)
    return dedup[:6]


def _extract_tags(anime):
    tags = anime.get("tags")
    if not isinstance(tags, list):
        return []
    ranked = []
    for tag in tags:
        if isinstance(tag, dict):
            name = _as_nonempty_str(tag.get("name"))
            rank = tag.get("rank")
            try:
                rank_val = int(rank) if rank is not None else 0
            except Exception:
                rank_val = 0
            if name and rank_val >= 60:
                ranked.append((name, rank_val))
        elif isinstance(tag, str):
            ranked.append((tag.strip(), 60))
    ranked.sort(key=lambda x: x[1], reverse=True)
    names = []
    seen = set()
    for name, _ in ranked:
        key = name.lower()
        if key in seen:
            continue
        seen.add(key)
        names.append(name)
    return names[:20]


def _extract_studios(anime):
    studios = anime.get("studios")
    out = []
    if isinstance(studios, list):
        for studio in studios:
            if isinstance(studio, dict):
                name = _as_nonempty_str(studio.get("name"))
                if name:
                    out.append(name)
            elif isinstance(studio, str):
                name = _as_nonempty_str(studio)
                if name:
                    out.append(name)
    dedup = []
    seen = set()
    for name in out:
        key = name.lower()
        if key in seen:
            continue
        seen.add(key)
        dedup.append(name)
    return dedup[:3]


def _extract_relations(anime):
    relations = anime.get("relations")
    titles = []
    if isinstance(relations, list):
        for rel in relations:
            if not isinstance(rel, dict):
                continue
            rel_title = rel.get("title")
            if isinstance(rel_title, dict):
                preferred = _as_nonempty_str(rel_title.get("english")) or _as_nonempty_str(rel_title.get("romaji"))
                if preferred:
                    titles.append(preferred)
            elif isinstance(rel_title, str):
                title = _as_nonempty_str(rel_title)
                if title:
                    titles.append(title)
    dedup = []
    seen = set()
    for title in titles:
        key = title.lower()
        if key in seen:
            continue
        seen.add(key)
        dedup.append(title)
    return dedup[:5]


def _normalize_for_dedupe(text):
    return " ".join("".join(ch.lower() if ch.isalnum() else " " for ch in text).split())


def _is_near_duplicate(norm_text, existing_norm):
    if not norm_text or not existing_norm:
        return True
    if norm_text == existing_norm:
        return True
    shorter = min(len(norm_text), len(existing_norm))
    longer = max(len(norm_text), len(existing_norm))
    if shorter > 0 and (shorter / longer) >= 0.9 and (
        norm_text in existing_norm or existing_norm in norm_text
    ):
        return True
    return difflib.SequenceMatcher(None, norm_text, existing_norm).ratio() >= 0.96


def _dedupe_reviews_with_meta(review_meta):
    deduped = []
    normalized = []
    for item in review_meta:
        text = _as_nonempty_str(item.get("text") if isinstance(item, dict) else item)
        if not text:
            continue
        norm = _normalize_for_dedupe(text)
        if any(_is_near_duplicate(norm, prev) for prev in normalized):
            continue
        normalized.append(norm)
        if isinstance(item, dict):
            deduped.append(item)
        else:
            deduped.append({"text": text, "mask_applied": True})
    return deduped


def _contains_unmasked_title(title, selected_review_meta):
    title_clean = _as_nonempty_str(title)
    if not title_clean:
        return False
    title_lower = title_clean.lower()
    title_head = title_lower.split(":", 1)[0].strip()
    for item in selected_review_meta:
        if item.get("mask_applied"):
            continue
        text_lower = str(item.get("text", "")).lower()
        if title_lower in text_lower or (title_head and title_head in text_lower):
            return True
    return False


def build_corpus_entry(anime):
    anilist_id = anime["anilist_id"]
    mal_id = anime.get("mal_id")

    sections = []
    section_names = []

    title = _as_nonempty_str(anime.get("title")) or f"AniList:{anilist_id}"
    sections.append(f"Title: {title}")
    section_names.append("title")

    aliases = _extract_aliases(anime)
    if aliases:
        sections.append(f"Aliases: {', '.join(aliases)}")
        section_names.append("aliases")

    format_value = _as_nonempty_str(anime.get("format"))
    season = _as_nonempty_str(anime.get("season"))
    season_year = anime.get("season_year")
    status = _as_nonempty_str(anime.get("status"))
    lifecycle_parts = []
    if format_value:
        lifecycle_parts.append(f"Format={format_value}")
    if season:
        season_part = f"Season={season}"
        if season_year is not None:
            season_part += f" {season_year}"
        lifecycle_parts.append(season_part)
    if status:
        lifecycle_parts.append(f"Status={status}")
    if lifecycle_parts:
        sections.append("Lifecycle: " + ", ".join(lifecycle_parts))
        section_names.append("lifecycle")

    genres = _normalize_str_list(anime.get("genres"))
    if genres:
        sections.append(f"Genres: {', '.join(genres)}")
        section_names.append("genres")

    tags = _extract_tags(anime)
    if tags:
        sections.append(f"Tags: {', '.join(tags)}")
        section_names.append("tags")

    studios = _extract_studios(anime)
    if studios:
        sections.append(f"Studios: {', '.join(studios)}")
        section_names.append("studios")

    relations = _extract_relations(anime)
    if relations:
        sections.append(f"Relations: {', '.join(relations)}")
        section_names.append("relations")

    if mal_id and mal_id in synopsis_by_mal:
        synopsis = _as_nonempty_str(synopsis_by_mal[mal_id])
        if synopsis:
            sections.append(f"Synopsis: {synopsis[:SYNOPSIS_CHAR_CAP]}")
            section_names.append("synopsis")

    review_meta = review_meta_by_anime.get(anilist_id, [])
    deduped_review_meta = _dedupe_reviews_with_meta(review_meta)
    selected_review_meta = deduped_review_meta[:MAX_REVIEWS_PER_ANIME]
    review_count_used = len(selected_review_meta)

    if selected_review_meta:
        review_body = "\n\n".join(item.get("text", "") for item in selected_review_meta if item.get("text"))
        remaining_budget = MAX_CHARS - sum(len(s) for s in sections)
        if remaining_budget > 200:
            sections.append(f"Reviews:\n{review_body[:remaining_budget]}")
            section_names.append("reviews")

    text = "\n".join(sections)[:MAX_CHARS]
    contains_unmasked_title = _contains_unmasked_title(title, selected_review_meta)

    return {
        "anilist_id": anilist_id,
        "mal_id": mal_id,
        "title": title,
        "text": text,
        "text_sections": section_names,
        "review_count_used": review_count_used,
        "contains_unmasked_title": contains_unmasked_title,
    }


# Build corpus
corpus = []
for anime in tqdm(anime_metadata, desc="Building corpus"):
    corpus.append(build_corpus_entry(anime))

print(f"Corpus entries: {len(corpus):,}")
text_lengths = [len(c["text"]) for c in corpus]
print(f"Text length: min={min(text_lengths)}, median={np.median(text_lengths):.0f}, max={max(text_lengths)}")



Building corpus:   0%|          | 0/5000 [00:00<?, ?it/s]

Corpus entries: 5,000
Text length: min=88, median=1862, max=9000


In [6]:
# Save corpus
with open(DATA_DIR / "corpus.jsonl", "w", encoding="utf-8") as f:
    for entry in tqdm(corpus, desc="Saving corpus"):
        f.write(json.dumps(entry) + "\n")

print(f"Saved corpus to {DATA_DIR / 'corpus.jsonl'}")

Saving corpus:   0%|          | 0/5000 [00:00<?, ?it/s]

Saved corpus to data\corpus.jsonl


## 2. Triplet Generation

For triplet loss training:
- **Anchor**: A review of anime X
- **Positive**: A different review of the same anime X
- **Negative**: A review of a different anime Y (hard negative: same genre but different series)

We need anime with 2+ reviews to form anchor/positive pairs.

In [7]:
# Build indices for hard negative mining
genre_to_anime = defaultdict(set)
tag_to_anime = defaultdict(set)
relation_to_anime = defaultdict(set)
anime_genres = {}
anime_tags = {}
anime_relations = {}
anime_popularity_bucket = {}


def _extract_top_tags(anime):
    tags = anime.get("tags")
    out = set()
    if isinstance(tags, list):
        for tag in tags:
            if isinstance(tag, dict):
                name = tag.get("name")
                rank = tag.get("rank")
                try:
                    rank_val = int(rank) if rank is not None else 0
                except Exception:
                    rank_val = 0
                if isinstance(name, str) and name.strip() and rank_val >= 60:
                    out.add(name.strip().lower())
            elif isinstance(tag, str) and tag.strip():
                out.add(tag.strip().lower())
    return out


def _extract_relation_ids(anime):
    relation_ids = set()
    relations = anime.get("relations")
    if isinstance(relations, list):
        for rel in relations:
            if not isinstance(rel, dict):
                continue
            rel_id = rel.get("id")
            if isinstance(rel_id, int):
                relation_ids.add(rel_id)
            elif isinstance(rel_id, str) and rel_id.isdigit():
                relation_ids.add(int(rel_id))
    return relation_ids


pop_values = [
    int(a.get("popularity"))
    for a in anime_metadata
    if isinstance(a.get("popularity"), (int, float)) and int(a.get("popularity")) > 0
]
if pop_values:
    p25, p50, p75 = np.quantile(pop_values, [0.25, 0.50, 0.75])
else:
    p25 = p50 = p75 = 0


def _pop_bucket(pop):
    if pop is None:
        return "unknown"
    try:
        val = int(pop)
    except Exception:
        return "unknown"
    if val <= 0:
        return "unknown"
    if val >= p75:
        return "very_popular"
    if val >= p50:
        return "popular"
    if val >= p25:
        return "mid"
    return "niche"


for anime in tqdm(anime_metadata, desc="Indexing genres/tags/relations"):
    aid = anime["anilist_id"]
    genres = {g.strip().lower() for g in anime.get("genres", []) if isinstance(g, str) and g.strip()}
    tags = _extract_top_tags(anime)
    relations = _extract_relation_ids(anime)

    anime_genres[aid] = genres
    anime_tags[aid] = tags
    anime_relations[aid] = relations
    anime_popularity_bucket[aid] = _pop_bucket(anime.get("popularity"))

    for g in genres:
        genre_to_anime[g].add(aid)
    for t in tags:
        tag_to_anime[t].add(aid)
    for rid in relations:
        relation_to_anime[rid].add(aid)

# Only consider anime with 2+ reviews for triplets
multi_review_anime = {aid: revs for aid, revs in reviews_by_anime.items() if len(revs) >= 2}
print(f"Anime with 2+ reviews (triplet candidates): {len(multi_review_anime):,}")
print(f"Total reviews in those anime: {sum(len(v) for v in multi_review_anime.values()):,}")



Indexing genres/tags/relations:   0%|          | 0/5000 [00:00<?, ?it/s]

Anime with 2+ reviews (triplet candidates): 1,838
Total reviews in those anime: 11,469


In [8]:
NEGATIVE_TYPE_WEIGHTS = {
    "partial_match": 0.50,
    "popularity_distractor": 0.30,
    "franchise_near_wrong_intent": 0.20,
}


def _weighted_pick(available_types):
    weights = [NEGATIVE_TYPE_WEIGHTS[t] for t in available_types]
    return random.choices(available_types, weights=weights, k=1)[0]


def _genre_overlap(aid_left, aid_right):
    return len(anime_genres.get(aid_left, set()) & anime_genres.get(aid_right, set()))


def _tag_overlap(aid_left, aid_right):
    return len(anime_tags.get(aid_left, set()) & anime_tags.get(aid_right, set()))


def _build_negative_pools(anchor_anime_id):
    anchor_genres = anime_genres.get(anchor_anime_id, set())
    anchor_tags = anime_tags.get(anchor_anime_id, set())
    anchor_relations = anime_relations.get(anchor_anime_id, set())
    anchor_bucket = anime_popularity_bucket.get(anchor_anime_id, "unknown")

    candidate_ids = set()
    for g in anchor_genres:
        candidate_ids |= genre_to_anime[g]
    for t in anchor_tags:
        candidate_ids |= tag_to_anime[t]
    for rid in anchor_relations:
        candidate_ids |= relation_to_anime[rid]
    candidate_ids.discard(anchor_anime_id)

    pools = {
        "partial_match": [],
        "popularity_distractor": [],
        "franchise_near_wrong_intent": [],
    }

    for cand_id in candidate_ids:
        if cand_id not in multi_review_anime:
            continue

        genre_overlap = _genre_overlap(anchor_anime_id, cand_id)
        tag_overlap = _tag_overlap(anchor_anime_id, cand_id)
        rel_overlap = len(anchor_relations & anime_relations.get(cand_id, set()))
        cand_bucket = anime_popularity_bucket.get(cand_id, "unknown")

        if genre_overlap >= 1 and tag_overlap >= 1 and genre_overlap <= 2:
            pools["partial_match"].append((cand_id, genre_overlap, tag_overlap))

        if genre_overlap >= 1 and cand_bucket != anchor_bucket and cand_bucket != "unknown":
            pools["popularity_distractor"].append((cand_id, genre_overlap, tag_overlap))

        if rel_overlap >= 1 and genre_overlap <= 1:
            pools["franchise_near_wrong_intent"].append((cand_id, genre_overlap, tag_overlap))

    return pools


def get_hard_negative(anchor_anime_id, forbidden_texts):
    pools = _build_negative_pools(anchor_anime_id)
    available_types = [k for k, vals in pools.items() if vals]

    for t in available_types:
        triplet_generation_stats["negative_type_available_counts"][t] += 1

    for _ in range(6):
        if available_types:
            neg_type = _weighted_pick(available_types)
            cand_id, genre_overlap, tag_overlap = random.choice(pools[neg_type])
        else:
            fallback = [a for a in multi_review_anime if a != anchor_anime_id]
            if not fallback:
                return None
            neg_type = "fallback_random"
            cand_id = random.choice(fallback)
            genre_overlap = _genre_overlap(anchor_anime_id, cand_id)
            tag_overlap = _tag_overlap(anchor_anime_id, cand_id)

        candidate_reviews = multi_review_anime.get(cand_id, [])
        if not candidate_reviews:
            continue
        negative_text = random.choice(candidate_reviews)
        if negative_text in forbidden_texts:
            continue

        return {
            "negative_text": negative_text,
            "negative_anime_id": cand_id,
            "negative_type": neg_type,
            "genre_overlap": genre_overlap,
            "tag_overlap": tag_overlap,
        }

    return None


# Generate triplets
MAX_REVIEW_LEN = 768
MAX_TRIPLETS_PER_ANIME = 14
MIN_REVIEW_CHARS = 140
MIN_DISTINCT_NEGATIVES_PER_ANIME = 3

triplets = []
triplet_generation_stats = {
    "negative_type_counts": defaultdict(int),
    "negative_type_available_counts": defaultdict(int),
    "genre_overlap_sum": defaultdict(float),
    "tag_overlap_sum": defaultdict(float),
    "type_pair_count": defaultdict(int),
    "distinct_negative_shortfalls": 0,
}

for anime_id, rev_list in tqdm(multi_review_anime.items(), total=len(multi_review_anime), desc="Generating triplets"):
    rev_list = [r for r in rev_list if len(r) >= MIN_REVIEW_CHARS]
    if len(rev_list) < 2:
        continue

    triplets_for_anime = 0
    distinct_negative_ids = set()

    for i in range(len(rev_list)):
        for j in range(len(rev_list)):
            if i == j:
                continue

            anchor = rev_list[i][:MAX_REVIEW_LEN]
            positive = rev_list[j][:MAX_REVIEW_LEN]
            forbidden = {anchor, positive}
            negative_info = get_hard_negative(anime_id, forbidden)
            if negative_info is None:
                continue

            negative = negative_info["negative_text"][:MAX_REVIEW_LEN]
            neg_type = negative_info["negative_type"]
            neg_aid = negative_info["negative_anime_id"]
            genre_overlap = int(negative_info["genre_overlap"])
            tag_overlap = int(negative_info["tag_overlap"])

            triplets.append({
                "anchor": anchor,
                "positive": positive,
                "negative": negative,
                "anchor_anime_id": anime_id,
                "negative_anime_id": neg_aid,
                "negative_type": neg_type,
                "genre_overlap": genre_overlap,
                "tag_overlap": tag_overlap,
            })

            distinct_negative_ids.add(neg_aid)
            triplet_generation_stats["negative_type_counts"][neg_type] += 1
            triplet_generation_stats["genre_overlap_sum"][neg_type] += genre_overlap
            triplet_generation_stats["tag_overlap_sum"][neg_type] += tag_overlap
            triplet_generation_stats["type_pair_count"][neg_type] += 1

            triplets_for_anime += 1
            if triplets_for_anime >= MAX_TRIPLETS_PER_ANIME:
                break
        else:
            continue
        break

    if len(distinct_negative_ids) < MIN_DISTINCT_NEGATIVES_PER_ANIME:
        triplet_generation_stats["distinct_negative_shortfalls"] += 1

random.shuffle(triplets)
print(f"Total triplets generated: {len(triplets):,}")
print("Negative type counts:", dict(triplet_generation_stats["negative_type_counts"]))



Generating triplets:   0%|          | 0/1838 [00:00<?, ?it/s]

Total triplets generated: 16,544
Negative type counts: {'popularity_distractor': 6213, 'partial_match': 10331}


In [9]:
# Save triplets
with open(DATA_DIR / "triplets.jsonl", "w", encoding="utf-8") as f:
    for t in triplets:
        f.write(json.dumps(t) + "\n")

print(f"Saved {len(triplets):,} triplets to {DATA_DIR / 'triplets.jsonl'}")

Saved 16,544 triplets to data\triplets.jsonl


## 3. CF Rating Matrix

Build a sparse user×anime matrix from filtered Kaggle ratings.
- Rows: users (indexed 0..N-1)
- Columns: anime (indexed 0..M-1)
- Values: normalized ratings (user mean subtracted)

Also store the anime index mapping so we can map predictions back to AniList IDs.

In [10]:
# Create index mappings
unique_users = sorted(ratings_df['user_id'].unique())
unique_anime = sorted(ratings_df['anime_id'].unique())  # MAL IDs

user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
anime_to_idx = {aid: i for i, aid in enumerate(unique_anime)}
idx_to_anime = {i: aid for aid, i in anime_to_idx.items()}

n_users = len(unique_users)
n_anime = len(unique_anime)

print(f"Matrix dimensions: {n_users:,} users × {n_anime:,} anime")
print(f"Sparsity: {1 - len(ratings_df) / (n_users * n_anime):.4%}")

Matrix dimensions: 265,263 users × 12,127 anime
Sparsity: 98.2225%


In [11]:
# Per-user mean normalization
user_means = ratings_df.groupby('user_id')['rating'].mean()

rows = []
cols = []
vals = []

for _, row in tqdm(ratings_df.iterrows(), total=len(ratings_df), desc="Building CF matrix"):
    uid = row['user_id']
    aid = row['anime_id']
    rating = row['rating']

    user_idx = user_to_idx[uid]
    anime_idx = anime_to_idx[aid]
    normalized = rating - user_means[uid]

    rows.append(user_idx)
    cols.append(anime_idx)
    vals.append(normalized)

cf_matrix = sparse.csr_matrix(
    (vals, (rows, cols)),
    shape=(n_users, n_anime)
)

print(f"Sparse matrix: {cf_matrix.shape}, nnz={cf_matrix.nnz:,}")
print(f"Density: {cf_matrix.nnz / (cf_matrix.shape[0] * cf_matrix.shape[1]):.4%}")

Building CF matrix:   0%|          | 0/57178041 [00:00<?, ?it/s]

Sparse matrix: (265263, 12127), nnz=57,178,041
Density: 1.7775%


In [12]:
# Save CF matrix and mappings
sparse.save_npz(DATA_DIR / "cf_ratings.npz", cf_matrix)

# Save anime index mapping (MAL ID -> index, with AniList ID where available)
anime_index = []
for mal_id, idx in anime_to_idx.items():
    anilist_id = mal_to_anilist.get(mal_id)
    anime_index.append({
        "idx": int(idx),
        "mal_id": int(mal_id),
        "anilist_id": int(anilist_id) if anilist_id is not None else None
    })

with open(DATA_DIR / "cf_anime_index.json", "w", encoding="utf-8") as f:
    json.dump(anime_index, f)

# Save user means (needed for denormalization at inference)
user_means_dict = {str(uid): float(mean) for uid, mean in user_means.items()}
with open(DATA_DIR / "user_means.json", "w", encoding="utf-8") as f:
    json.dump(user_means_dict, f)

print(f"Saved CF matrix: {DATA_DIR / 'cf_ratings.npz'}")
print(f"Saved anime index: {DATA_DIR / 'cf_anime_index.json'} ({len(anime_index)} entries)")
print(f"Saved user means: {DATA_DIR / 'user_means.json'}")

Saved CF matrix: data\cf_ratings.npz
Saved anime index: data\cf_anime_index.json (12127 entries)
Saved user means: data\user_means.json


## 4. Summary

In [13]:
from datetime import datetime, timezone

print("=" * 50)
print("PREPROCESSING SUMMARY")
print("=" * 50)

print(f"\n[Semantic Corpus]")
print(f"  Entries: {len(corpus):,}")
print(f"  Avg text length: {np.mean(text_lengths):.0f} chars")

print(f"\n[Triplets]")
print(f"  Count: {len(triplets):,}")

print(f"\n[CF Matrix]")
print(f"  Shape: {cf_matrix.shape}")
print(f"  Sparsity: {(1 - cf_matrix.nnz / (cf_matrix.shape[0] * cf_matrix.shape[1])) * 100:.2f}%")

print(f"\n[Outputs]")
print(f"  [OK] {DATA_DIR / 'corpus.jsonl'}")
print(f"  [OK] {DATA_DIR / 'triplets.jsonl'}")
print(f"  [OK] {DATA_DIR / 'cf_ratings.npz'}")
print(f"  [OK] {DATA_DIR / 'cf_anime_index.json'}")
print(f"  [OK] {DATA_DIR / 'user_means.json'}")


def _coverage(section_name):
    if not corpus:
        return 0.0
    count = sum(1 for row in corpus if section_name in row.get("text_sections", []))
    return count / len(corpus)


def _safe_share(num, den):
    return 0.0 if den <= 0 else num / den


def _safe_avg(num, den):
    return 0.0 if den <= 0 else num / den


section_coverage = {
    "aliases": _coverage("aliases"),
    "tags": _coverage("tags"),
    "studios": _coverage("studios"),
    "relations": _coverage("relations"),
    "synopsis": _coverage("synopsis"),
    "reviews": _coverage("reviews"),
}

kept_reviews = int(review_diag_totals.get("kept_reviews", 0))
dropped_short = int(review_diag_totals.get("dropped_short", 0))
dropped_noise = int(review_diag_totals.get("dropped_noise", 0))
total_reviews = int(review_diag_totals.get("total_reviews", 0))
reviews_kept_rate = _safe_share(kept_reviews, total_reviews)
masking_ratio_actual = _safe_share(review_diag_totals.get("masked_reviews", 0), total_reviews)

negative_counts = dict(triplet_generation_stats["negative_type_counts"])
negative_available = dict(triplet_generation_stats["negative_type_available_counts"])
negative_total = sum(negative_counts.values())
negative_distribution = {
    k: _safe_share(v, negative_total)
    for k, v in negative_counts.items()
}

avg_genre_overlap = {
    k: _safe_avg(
        triplet_generation_stats["genre_overlap_sum"].get(k, 0.0),
        triplet_generation_stats["type_pair_count"].get(k, 0),
    )
    for k in negative_counts
}
avg_tag_overlap = {
    k: _safe_avg(
        triplet_generation_stats["tag_overlap_sum"].get(k, 0.0),
        triplet_generation_stats["type_pair_count"].get(k, 0),
    )
    for k in negative_counts
}

thresholds = {
    "reviews_kept_rate_min": 0.60,
    "aliases_section_coverage_min": 0.70,
    "tags_section_coverage_min": 0.90,
    "negative_type_min_share": 0.10,
}

failures = []
if reviews_kept_rate < thresholds["reviews_kept_rate_min"]:
    failures.append(
        f"reviews_kept_rate {reviews_kept_rate:.4f} < {thresholds['reviews_kept_rate_min']:.2f}"
    )
if section_coverage["aliases"] < thresholds["aliases_section_coverage_min"]:
    failures.append(
        f"aliases coverage {section_coverage['aliases']:.4f} < {thresholds['aliases_section_coverage_min']:.2f}"
    )
if section_coverage["tags"] < thresholds["tags_section_coverage_min"]:
    failures.append(
        f"tags coverage {section_coverage['tags']:.4f} < {thresholds['tags_section_coverage_min']:.2f}"
    )

for ntype in ("partial_match", "popularity_distractor", "franchise_near_wrong_intent"):
    available_count = int(negative_available.get(ntype, 0))
    share = float(negative_distribution.get(ntype, 0.0))
    if available_count > 0 and share < thresholds["negative_type_min_share"]:
        failures.append(
            f"negative_type '{ntype}' share {share:.4f} < {thresholds['negative_type_min_share']:.2f}"
        )

report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "corpus": {
        "count": len(corpus),
        "avg_text_length": float(np.mean(text_lengths)) if text_lengths else 0.0,
        "section_coverage": section_coverage,
    },
    "review_preprocessing": {
        "total_reviews": total_reviews,
        "kept_reviews": kept_reviews,
        "dropped_short": dropped_short,
        "dropped_noise": dropped_noise,
        "reviews_kept_rate": reviews_kept_rate,
        "masking_ratio_target": MASK_TITLE_PROBABILITY,
        "masking_ratio_actual": masking_ratio_actual,
        "sentence_stats": {
            "total_sentences": int(review_diag_totals.get("total_sentences", 0)),
            "noise_filtered": int(review_diag_totals.get("noise_filtered", 0)),
            "low_score_filtered": int(review_diag_totals.get("low_score_filtered", 0)),
            "kept_sentences": int(review_diag_totals.get("kept_sentences", 0)),
        },
    },
    "triplets": {
        "count": len(triplets),
        "negative_type_counts": negative_counts,
        "negative_type_distribution": negative_distribution,
        "negative_type_available_counts": negative_available,
        "avg_genre_overlap": avg_genre_overlap,
        "avg_tag_overlap": avg_tag_overlap,
        "distinct_negative_shortfalls": int(triplet_generation_stats.get("distinct_negative_shortfalls", 0)),
    },
    "thresholds": thresholds,
    "gate_pass": len(failures) == 0,
    "failures": failures,
}

EVAL_DIR = Path("eval")
EVAL_DIR.mkdir(parents=True, exist_ok=True)
report_path = EVAL_DIR / f"preprocessing_quality_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(f"\n[Preprocessing Diagnostics]")
print(f"  Report: {report_path}")
print(f"  Gate pass: {report['gate_pass']}")
if failures:
    print("  Failures:")
    for failure in failures:
        print(f"    - {failure}")
    raise RuntimeError("Preprocessing quality gates failed; fix data quality before training.")



PREPROCESSING SUMMARY

[Semantic Corpus]
  Entries: 5,000
  Avg text length: 2839 chars

[Triplets]
  Count: 16,544

[CF Matrix]
  Shape: (265263, 12127)
  Sparsity: 98.22%

[Outputs]
  [OK] data\corpus.jsonl
  [OK] data\triplets.jsonl
  [OK] data\cf_ratings.npz
  [OK] data\cf_anime_index.json
  [OK] data\user_means.json

[Preprocessing Diagnostics]
  Report: eval\preprocessing_quality_20260227T234716Z.json
  Gate pass: True
